# NVT de pw.x → xyz (por caso y conjunto etiquetado)

Versión en notebook de `nvt2xyz.py`. Lee los `output.N` de las dinámicas NVT de Quantum ESPRESSO y escribe:

- un `<caso>_T<T>K.xyz` por caso, con **todos** los pasos MD;
- `nvt_vacantes_todas.xyz` con todos los casos juntos y etiquetados. Es la entrada del notebook de análisis `BGDY_3x3_vacancies_postNVT.ipynb`.

**Entrada:** una carpeta con un subdirectorio por caso que contenga sus `output.1`, `output.2`… (se busca también en subcarpetas, así que vale el árbol `cases/` del workflow). Opcionalmente, la carpeta `structures/` de `make_vacancies.py`, que da la celda exacta, el número de vacantes, los casos de test y la distancia de cada átomo a la vacante (`d_vac`).

```
dataset_exportat/                structures/
├── case_01_V1_alpha/            ├── case_00_pristine.xyz
│   ├── output.1                 ├── case_01_V1_alpha.xyz ...
│   └── output.2 ...             └── manifest.csv
└── case_02_V1_beta/ ...
```

Qué hace con cada NVT: salta (diciendo por qué) los outputs sin ningún paso completo o con otros átomos; une los reinicios por número de paso, con la geometría correcta en el primer paso de cada reinicio; comprueba el salto de energía en cada unión; y escribe el xyz. Solo usa la librería estándar de Python.

Cada frame lleva: `energy` (eV), `forces` (eV/Å), `stress` (eV/Å³, signo de ASE), posiciones en Å sin envolver, `case`, `case_num`, `n_vac`, `T_K`, `step`, `time_fs`, `dt_fs`, `src` (n.º de output), `T_inst_K`, `mag_tot`, `mag_abs`, `split_sugerido`, `nvt_completa` y, por átomo, `d_vac` (Å).

## 1. Parámetros

In [ ]:
from pathlib import Path

CARPETA_ENTRADA = Path("dataset")      # subcarpetas case_NN_*/ con los output.N
CARPETA_STRUCTURES = Path("structures")         # None si no la tienes (el xyz saldrá sin d_vac ni casos de test)
CARPETA_SALIDA = Path("dataset_xyz")   # se crea si no existe
NOMBRE_CONJUNTO = "nvt_vacantes_todas.xyz"      # fichero con todos los casos, dentro de CARPETA_SALIDA

CASOS = None          # None = todos los de la carpeta; o una lista: [1, 2, 6], ["10-12"], ["missing-linker"]...
TEMP_DEFECTO = None   # T (K) si no se puede deducir del nombre de la carpeta (nvt_T1000K) ni del output

## 2. Funciones

Son las mismas de `nvt2xyz.py` (parser verificado con salidas reales de pw.x). No hace falta tocarlas.

In [ ]:
import csv
import re
from pathlib import Path

RY_EV = 13.605693122994
BOHR_A = 0.529177210903
RY_AU_FS = 0.0483776865          # unidad de tiempo de pw.x (u.a. de Rydberg) en fs
F_UNIT = RY_EV / BOHR_A           # Ry/bohr -> eV/A
S_UNIT = RY_EV / BOHR_A ** 3      # Ry/bohr^3 -> eV/A^3
FL = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[EeDd][-+]?\d+)?")
TAU_RE = re.compile(r"\s*\d+\s+(\S+)\s+tau\(\s*\d+\)\s*=\s*\((.*)\)")
TEMP_RE = re.compile(r"temperature\s*=\s*([-+0-9.EeDd]+)\s*K")


def _f(s):
    return [float(x.replace("D", "E").replace("d", "e")) for x in FL.findall(s)]


def matmul_rows(rows, cell):
    return [[sum(r[i] * cell[i][k] for i in range(3)) for k in range(3)] for r in rows]


def leer_pw_md(path):
    """Lee un output de pw.x (calculation='md'). Cada SCF lleva step = ultima iteracion vista
    (None para el primero del fichero; se resuelve al unir)."""
    nat = alat = cell = species = pos0 = dt_fs = species_pos = None
    frames, positions, temps = [], {}, {}
    it = first_it = None
    E = mag = amag = None
    nonconv, done, error = 0, False, None
    with open(str(path), errors="replace") as fh:
        for ln in fh:
            s = ln.strip()
            if not s:
                continue
            if s.startswith("!") and "total energy" in s:
                E, mag, amag = _f(s.split("=")[1])[0] * RY_EV, None, None
            elif s.startswith("total magnetization") and E is not None and mag is None:
                mag = _f(s.split("=")[1])[0]
            elif s.startswith("absolute magnetization") and E is not None and amag is None:
                amag = _f(s.split("=")[1])[0]
            elif s.startswith("convergence NOT achieved"):
                nonconv += 1
                E = None
            elif s.startswith("Forces acting on atoms"):
                F = []
                for l2 in fh:
                    t = l2.strip()
                    if not t:
                        if F:
                            break
                        continue
                    if not t.startswith("atom") or "force" not in t:
                        break
                    F.append([v * F_UNIT for v in _f(t.split("=")[1])[:3]])
                    if len(F) == nat:
                        break
                if E is not None and len(F) == nat:
                    frames.append({"step": it, "E": E, "F": F, "S": None, "mag": mag, "amag": amag})
                E = None
            elif s.startswith("total   stress"):
                rows = []
                for l2 in fh:
                    v = _f(l2)
                    if len(v) < 3:
                        break
                    rows.append([x * S_UNIT for x in v[:3]])
                    if len(rows) == 3:
                        break
                if len(rows) == 3 and frames and frames[-1]["S"] is None and frames[-1]["step"] == it:
                    frames[-1]["S"] = rows
            elif s.startswith("Entering Dynamics"):
                it = int(s.split("=")[1])
                if first_it is None:
                    first_it = it
            elif s.startswith("ATOMIC_POSITIONS"):
                unit = s.lower()
                sp, xyz = [], []
                for l2 in fh:
                    t = l2.split()
                    if len(t) < 4 or not t[0][0].isalpha():
                        break
                    sp.append(t[0])
                    xyz.append(_f(" ".join(t[1:]))[:3])
                    if len(sp) == nat:
                        break
                if len(sp) == nat:
                    if species_pos is None:
                        species_pos = sp
                    if "bohr" in unit:
                        xyz = [[c * BOHR_A for c in r] for r in xyz]
                    elif "crystal" in unit:
                        xyz = matmul_rows(xyz, cell)
                    elif "alat" in unit:
                        xyz = [[c * alat for c in r] for r in xyz]
                    positions[it if it is not None else -1] = xyz
            elif s.startswith("temperature") and it is not None:
                m = TEMP_RE.match(s)
                if m:
                    temps[it] = float(m.group(1))
            elif s.startswith("Time step") and dt_fs is None:
                m = re.search(r"=\s*([\d.]+)\s*a\.u\.", s)
                if m:
                    dt_fs = float(m.group(1)) * RY_AU_FS
            elif nat is None and s.startswith("number of atoms/cell"):
                nat = int(s.split("=")[1])
            elif alat is None and s.startswith("celldm(1)="):
                alat = _f(s.split("=")[1])[0] * BOHR_A
            elif cell is None and s.startswith("crystal axes"):
                cell = [[c * alat for c in _f(next(fh).split("=")[1])[:3]] for _ in range(3)]
            elif pos0 is None and "positions (alat units)" in s:
                sp, xyz = [], []
                for l2 in fh:
                    m = TAU_RE.match(l2)
                    if not m:
                        break
                    sp.append(m.group(1))
                    xyz.append([c * alat for c in _f(m.group(2))[:3]])
                species, pos0 = sp, xyz
            elif s.startswith("JOB DONE"):
                done = True
            elif s.startswith("Error in routine") and error is None:
                error = s + " " + next(fh, "").strip()
    return {"path": str(path), "nat": nat, "cell": cell, "species": species, "species_pos": species_pos,
            "pos0": pos0, "dt_fs": dt_fs,
            "frames": frames, "positions": positions, "temps": temps, "first_it": first_it,
            "nonconv": nonconv, "done": done, "error": error}



def diagnostico(P):
    """None si el output se puede usar; si no, el motivo."""
    if P["nat"] is None:
        return "no parece un output de pw.x (vacio o cortado antes de la cabecera)"
    if not P["frames"]:
        leidos = len(P["species"] or [])
        cab = "" if leidos == P["nat"] else "cabecera incompleta (%d/%d atomos), " % (leidos, P["nat"])
        err = "; error: " + P["error"] if P["error"] else ""
        return "%sningun paso MD completo -> trabajo recien arrancado o cortado al empezar?%s" % (cab, err)
    if P["species"] is None or len(P["species"]) != P["nat"]:
        P["species"] = P["species_pos"]
    if P["cell"] is None:
        return "sin celda en la cabecera"
    return None


def unir_outputs(parsed):
    """parsed: outputs ya validados, en orden. Devuelve (trayectoria, tabla de outputs)."""
    acc, pos_map, temps, info = {}, {}, {}, []
    for P in parsed:
        recs, missing = [], 0
        for fr in P["frames"]:
            st = fr["step"]
            if st is None:
                if P["first_it"] is not None:
                    st = P["first_it"] - 1
                else:
                    st = max(pos_map) if pos_map else 0
            if st == 0 and P["first_it"] in (None, 1) and P["pos0"] is not None:
                pos = P["pos0"]
            elif st in P["positions"]:
                pos = P["positions"][st]
            elif st in pos_map:
                pos = pos_map[st]
            else:
                missing += 1
                continue
            r = dict(fr)
            r.update(step=st, pos=pos, src=P["n"])
            recs.append(r)
        superseded = 0
        if recs:
            smin = min(r["step"] for r in recs)
            for s in [s for s in acc if s >= smin]:
                del acc[s]
                superseded += 1
            for r in recs:
                acc[r["step"]] = r
        pos_map.update(dict((s, v) for s, v in P["positions"].items() if s >= 0))
        temps.update(P["temps"])
        steps_k = [r["step"] for r in recs]
        info.append({"output": Path(P["path"]).name, "usado": True, "motivo": "", "scf": len(P["frames"]),
                     "pasos": len(recs), "desde": min(steps_k) if steps_k else "",
                     "hasta": max(steps_k) if steps_k else "", "sustituidos": superseded,
                     "sin_geometria": missing, "scf_no_conv": P["nonconv"], "job_done": P["done"],
                     "error": P["error"] or "", "salto_E_union": ""})
    steps = sorted(acc)
    recs = [acc[s] for s in steps]
    for r in recs:
        r["T"] = temps.get(r["step"])
    dE = [abs(recs[j + 1]["E"] - recs[j]["E"]) for j in range(len(recs) - 1)]
    if dE:
        med = sorted(dE)[len(dE) // 2]
        by_n = dict((Path(P["path"]).name, k) for k, P in enumerate(parsed))
        for j in range(len(recs) - 1):
            if recs[j + 1]["src"] != recs[j]["src"] and med > 0:
                k = by_n["output.%d" % recs[j + 1]["src"]]
                info[k]["salto_E_union"] = round(dE[j] / med, 2)
    return {"species": parsed[0]["species"], "cell": parsed[0]["cell"], "recs": recs}, info



# ------------------------------------------------------------------ estructuras de referencia
def leer_xyz(path):
    L = Path(path).read_text().splitlines()
    n = int(L[0].split()[0])
    m = re.search(r'Lattice\s*=\s*"([^"]+)"', L[1])
    v = [float(x) for x in m.group(1).split()] if m else None
    cell = [v[0:3], v[3:6], v[6:9]] if v else None
    sp, pos = [], []
    for ln in L[2:2 + n]:
        t = ln.split()
        sp.append(t[0])
        pos.append([float(t[1]), float(t[2]), float(t[3])])
    return sp, pos, cell


def inv3(a):
    det = (a[0][0] * (a[1][1] * a[2][2] - a[1][2] * a[2][1]) - a[0][1] * (a[1][0] * a[2][2] - a[1][2] * a[2][0])
           + a[0][2] * (a[1][0] * a[2][1] - a[1][1] * a[2][0]))
    c = [[(a[(j + 1) % 3][(i + 1) % 3] * a[(j + 2) % 3][(i + 2) % 3] - a[(j + 1) % 3][(i + 2) % 3] * a[(j + 2) % 3][(i + 1) % 3]) / det
          for j in range(3)] for i in range(3)]
    return c


def dist_mic_plano(p, q, cell, icell):
    """Distancia minima con periodicidad en el plano (a, b) y sin ella en c."""
    d = [p[k] - q[k] for k in range(3)]
    f = [sum(d[k] * icell[k][i] for k in range(3)) for i in range(3)]
    f[0] -= round(f[0])
    f[1] -= round(f[1])
    best = None
    for i in (-1, 0, 1):            # celda hexagonal (60 grados): redondear no basta, se miran las 9 imagenes
        for j in (-1, 0, 1):
            g = (f[0] + i, f[1] + j, f[2])
            r = [sum(g[n] * cell[n][k] for n in range(3)) for k in range(3)]
            d2 = r[0] ** 2 + r[1] ** 2 + r[2] ** 2
            if best is None or d2 < best:
                best = d2
    return best ** 0.5


def d_vacante(xyz_caso, xyz_perfecta):
    """Distancia de cada atomo del caso al sitio vacante mas cercano (geometria inicial).
    Sitio vacante = atomo de la celda perfecta sin pareja (< 0.1 A) en el caso."""
    spc, pc, cell = leer_xyz(xyz_caso)
    _, pp, _ = leer_xyz(xyz_perfecta)
    icell = inv3(cell)
    vac = [p for p in pp if min(dist_mic_plano(p, q, cell, icell) for q in pc) > 0.1]
    if not vac:
        return spc, None
    return spc, [min(dist_mic_plano(q, v, cell, icell) for v in vac) for q in pc]


def manifest(sdir):
    out = {}
    f = Path(sdir) / "manifest.csv" if sdir else None
    if f and f.is_file():
        with open(str(f)) as fh:
            for r in csv.DictReader(fh):
                try:
                    out[int(r["case"])] = r
                except (KeyError, ValueError):
                    pass
    return out


# ------------------------------------------------------------------ escritura
def frames_xyz(traj, dt, etiquetas, d_vac):
    sp, cell = traj["species"], traj["cell"]
    lat = " ".join("%.10f" % v for row in cell for v in row)
    props = "species:S:1:pos:R:3:forces:R:3" + (":d_vac:R:1" if d_vac else "")
    fijo = " ".join("%s=%s" % kv for kv in etiquetas)
    for r in traj["recs"]:
        kv = ["energy=%.8f" % r["E"]]
        if r["S"] is not None:   # ASE usa el signo contrario al de QE
            kv.append('stress="%s"' % " ".join("%.10e" % (-v) for row in r["S"] for v in row))
        kv += [fijo, "step=%d" % r["step"], "time_fs=%.4f" % (r["step"] * dt), "src=%d" % r["src"]]
        if r["T"] is not None:
            kv.append("T_inst_K=%.3f" % r["T"])
        if r["mag"] is not None:
            kv.append("mag_tot=%.4f" % r["mag"])
        if r["amag"] is not None:
            kv.append("mag_abs=%.4f" % r["amag"])
        head = '%d\nLattice="%s" Properties=%s %s pbc="T T T"\n' % (len(sp), lat, props, " ".join(kv))
        if d_vac:
            body = "".join("%-2s %16.10f %16.10f %16.10f %15.8f %15.8f %15.8f %9.4f\n"
                           % (s, p[0], p[1], p[2], f[0], f[1], f[2], dv) for s, p, f, dv in zip(sp, r["pos"], r["F"], d_vac))
        else:
            body = "".join("%-2s %16.10f %16.10f %16.10f %15.8f %15.8f %15.8f\n"
                           % (s, p[0], p[1], p[2], f[0], f[1], f[2]) for s, p, f in zip(sp, r["pos"], r["F"]))
        yield head + body


# ------------------------------------------------------------------ una NVT
def outputs_de(nd):
    outs = []
    for f in Path(nd).glob("output.*"):
        m = re.match(r"output\.(\d+)$", f.name)
        if m:
            outs.append((int(m.group(1)), f))
    return sorted(outs)


def temperatura(nd, outs, defecto):
    m = re.search(r"nvt_T(\d+)K", nd.name)
    if m:
        return int(m.group(1))
    for _, f in outs:
        with open(str(f), errors="replace") as fh:
            for ln in fh:
                m = re.search(r"Starting temperature\s*=\s*([\d.]+)\s*K", ln)
                if m:
                    return int(round(float(m.group(1))))
                if "Entering Dynamics" in ln:
                    break
    return defecto


def contar_arranques(f):
    n = 0
    with open(str(f), errors="replace") as fh:
        for ln in fh:
            if "Program PWSCF" in ln and " starts on " in ln:
                n += 1
    return n


def procesar(nd, caso, sdir, man, temp_def):
    outs = outputs_de(nd)
    validos, lineas = [], {}
    for n, f in outs:
        P = leer_pw_md(f)
        P["n"] = n
        motivo = diagnostico(P)
        if motivo is None and validos and (P["nat"] != validos[0]["nat"] or P["species"] != validos[0]["species"]):
            ref = Path(validos[0]["path"]).name
            motivo = ("%d atomos, pero %s tiene %d" % (P["nat"], ref, validos[0]["nat"]) if P["nat"] != validos[0]["nat"]
                      else "mismos %d atomos que %s pero en otro orden (output de otro caso?)" % (P["nat"], ref))
        if motivo:
            arranques = contar_arranques(f)
            if arranques > 1 and not P["frames"]:
                motivo = ("%d arranques de pw.x en el mismo fichero, cada uno con 1 proceso MPI (srun lanzo copias "
                          "en serie: fallo del entorno MPI); ningun paso MD" % arranques)
            lineas[f.name] = ["%-10s SALTADO: %s" % (f.name, motivo)]
        else:
            validos.append(P)
    if not validos:
        for _, f in outs:
            print("   " + lineas[f.name][0])
        print("   ningun output utilizable")
        return None
    traj, info = unir_outputs(validos)
    for u in info:
        txt = "%-10s %5d pasos (%s-%s)%s" % (u["output"], u["pasos"], u["desde"], u["hasta"], "  JOB DONE" if u["job_done"] else "")
        if u["error"]:
            txt += "  [%s]" % u["error"][:70]
        if u["salto_E_union"] != "":
            txt += "  union: salto E x%.1f" % u["salto_E_union"]
        L = [txt]
        if u["salto_E_union"] != "" and u["salto_E_union"] > 10:
            L.append("AVISO: salto de energia muy grande en la union; revisa %s" % u["output"])
        if u["sustituidos"]:
            L.append("(empieza en el paso %s y sustituye %d pasos del output anterior)" % (u["desde"], u["sustituidos"]))
        if u["sin_geometria"]:
            L.append("AVISO: %d SCF sin geometria conocida (descartados)" % u["sin_geometria"])
        lineas[u["output"]] = L
    for _, f in outs:
        for k, t in enumerate(lineas[f.name]):
            print("   " + (t if k == 0 else " " * 11 + t))

    dt = next((P["dt_fs"] for P in validos if P["dt_fs"]), None)
    if dt is None:
        print("   ERROR: no encuentro 'Time step' en los outputs; no se exporta")
        return None
    T = temperatura(nd, outs, temp_def)
    num = int(re.match(r"case_(\d+)", caso).group(1)) if re.match(r"case_(\d+)", caso) else -1
    m = man.get(num, {})
    d_vac, avisos = None, []
    if sdir:
        xc = sorted(Path(sdir).glob("%s*.xyz" % caso)) or sorted(Path(sdir).glob("case_%02d_*.xyz" % num))
        xp = sorted(Path(sdir).glob("case_00_*.xyz"))
        if xc:
            sp0, _, cell0 = leer_xyz(xc[0])
            if sp0 != traj["species"]:
                avisos.append("el orden de atomos de %s no coincide con la NVT: sin d_vac" % xc[0].name)
            else:
                if cell0 and max(abs(a - b) for ra, rb in zip(cell0, traj["cell"]) for a, b in zip(ra, rb)) < 1e-3:
                    traj["cell"] = cell0
                if xp and num != 0:
                    _, d_vac = d_vacante(xc[0], xp[0])
        else:
            avisos.append("no encuentro %s*.xyz en %s: sin d_vac" % (caso, sdir))
    for a in avisos:
        print("   aviso: " + a)
    ultimo = validos[-1]
    etiquetas = [("case", caso), ("case_num", num), ("n_vac", int(m.get("n_vac", 0) or 0)), ("T_K", T),
                 ("dt_fs", "%.6f" % dt),
                 ("split_sugerido", "test" if "test" in m.get("description", "").lower() else "train"),
                 ("nvt_completa", "T" if (ultimo["done"] and not ultimo["error"]) else "F")]
    last = traj["recs"][-1]["step"]
    nv = sum(1 for x in d_vac if x < 4.0) if d_vac else 0
    print("   -> %d pasos (%d-%d), %.3f ps, dt %.4f fs, T %d K%s" % (
        len(traj["recs"]), traj["recs"][0]["step"], last, last * dt / 1000.0, dt, T,
        ", %d atomos a < 4 A de las vacantes" % nv if d_vac else ""))
    return traj, dt, etiquetas, d_vac, T

## 3. Ejecutar

In [ ]:
def elegido(caso, casos):
    """True si el caso entra en CASOS (números, rangos 'a-b', listas 'a,b' o trozos del nombre)."""
    if not casos:
        return True
    m = re.match(r"case_(\d+)", caso)
    num = int(m.group(1)) if m else None
    for tok in casos:
        for part in str(tok).split(","):
            part = part.strip()
            r = re.match(r"^(\d+)-(\d+)$", part)
            if r and num is not None and int(r.group(1)) <= num <= int(r.group(2)):
                return True
            if part.isdigit() and num == int(part):
                return True
            if part and not part.isdigit() and not r and part in caso:
                return True
    return False


raiz = Path(CARPETA_ENTRADA)
sdir = Path(CARPETA_STRUCTURES) if CARPETA_STRUCTURES else None
if not raiz.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de entrada {raiz.resolve()}")
if sdir and not sdir.is_dir():
    print(f"aviso: no existe {sdir.resolve()}; se sigue sin structures (sin d_vac ni casos de test)")
    sdir = None
man = manifest(sdir)
nvts = sorted(set(f.parent for f in raiz.rglob("output.*") if re.match(r"output\.\d+$", f.name)))
if not nvts:
    raise FileNotFoundError(f"No hay ficheros output.N bajo {raiz.resolve()}")

salida = Path(CARPETA_SALIDA)
salida.mkdir(parents=True, exist_ok=True)
conjunto = salida / NOMBRE_CONJUNTO
resumen = []
with open(conjunto, "w") as fc:
    for nd in nvts:
        rel = nd.relative_to(raiz).parts if nd != raiz else (nd.name,)
        caso = next((p for p in rel if re.match(r"case_\d+", p)), nd.name)
        if not elegido(caso, CASOS):
            continue
        print(f"{caso}  ({nd})")
        try:
            res = procesar(nd, caso, sdir, man, TEMP_DEFECTO)
        except Exception as e:  # una NVT rara no debe parar el resto
            print(f"   ERROR inesperado ({type(e).__name__}: {e}); se salta")
            res = None
        if not res:
            resumen.append(dict(caso=caso, T_K=None, frames=0, paso_ini=None, paso_fin=None, t_ps=None,
                                n_vac=None, split=None, fichero="(no exportado)"))
            continue
        traj, dt, etiquetas, d_vac, T = res
        fx = salida / f"{caso}_T{T}K.xyz"
        with open(fx, "w") as fh:
            for fr in frames_xyz(traj, dt, etiquetas, d_vac):
                fh.write(fr)
                fc.write(fr)
        et = dict(etiquetas)
        resumen.append(dict(caso=caso, T_K=T, frames=len(traj["recs"]), paso_ini=traj["recs"][0]["step"],
                            paso_fin=traj["recs"][-1]["step"], t_ps=round(traj["recs"][-1]["step"] * dt / 1000, 3),
                            n_vac=et["n_vac"], split=et["split_sugerido"], fichero=fx.name))

hechos = [r for r in resumen if r["frames"]]
print(f"\n{len(hechos)} NVT, {sum(r['frames'] for r in hechos)} frames")
print(f"xyz por caso : {salida.resolve()}")
print(f"xyz conjunto : {conjunto.resolve()} ({conjunto.stat().st_size / 1e6:.0f} MB)")
try:
    import pandas as pd
    display(pd.DataFrame(resumen))
except ImportError:
    for r in resumen:
        print(r)

## 4. Comprobación (opcional, necesita ASE)

Relee el xyz conjunto y cuenta los frames de cada caso.

In [ ]:
try:
    from ase.io import read
    from collections import Counter
    frames = read(conjunto, index=":", format="extxyz")
    print(f"{len(frames)} frames leídos de {conjunto.name}")
    for (c, t), n in sorted(Counter((a.info["case"], a.info["T_K"]) for a in frames).items()):
        print(f"   {c:32s} T={t} K  {n} frames")
    a = frames[0]
    print("etiquetas del primer frame:", {k: a.info[k] for k in a.info})
    print("propiedades por átomo:", list(a.arrays))
except ImportError:
    print("ASE no está instalado: pip install ase (solo para esta comprobación)")